# Phase 27 — Leakage claims: separating dataset evidence, our reproductions, and inferences about published models
**Resolves:** R2-9 (distinguish (a) direct dataset evidence, (b) reproduction experiments done here, (c) inferences about the original
published models; qualify claims where exact reconstruction was not possible) and the corresponding edits to Table 9/10 ("Yes (likely)").
Part A builds the tiered evidence table (numbers copied from the audit outputs of notebooks 02/05; verify against your notebooks).
Part B (optional) inspects the Pakistan oversampled file released with the original study to document, as far as the file allows, that the
oversampled data carry the same circularity. It is schema-defensive: if column names differ it prints the columns and stops without failing.
**Outputs:** `S2_leakage_evidence_tiers.csv`, `S2_pakistan_oversampled_inspection.csv` (if Part B runs)

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
tiers = [
 ("a) dataset fact", "Pakistan", "PHQ-9 item columns sum exactly to the supplied score in 100% of 14,008 rows (mean absolute difference 0)", "02_leakage_audit"),
 ("a) dataset fact", "Pakistan", "The binary label equals (PHQ-9 score >= 10) in every row", "02_leakage_audit"),
 ("a) dataset fact", "Arab-region", "PHQ-9 and GAD-7 items sum to the supplied scores (match rate 1.0) and the categories are exact thresholds of those scores", "02_leakage_audit"),
 ("a) dataset fact", "Arab-region", "Rows with missing items were default-imputed to the lowest-severity category by the data-collection process", "02_leakage_audit"),
 ("b) reproduction (this study)", "Pakistan", "Logistic regression on the 9 items -> label: 100% 5-fold CV accuracy; bare rule (score >= 10): 100%", "02_leakage_audit"),
 ("b) reproduction (this study)", "Arab-region", "Items -> label: 100% (depression), 99.7% (anxiety) cross-validated accuracy", "02_leakage_audit"),
 ("b) reproduction (this study)", "Arab-region", "Missingness of the PHQ-9 block predictable from SDoH variables (AUC ~0.96); full-sample SDoH model AUC 0.88-0.90 vs complete-item ~0.6-0.69", "02_leakage_audit / 05b"),
 ("b) reproduction (this study)", "Pakistan", "Validation-strategy contrast (ordinary vs group-aware CV) — numbers from Phase 17", "17_validation_strategy_comparison"),
 ("c) inference about published models", "Pakistan (Zafar et al.)", "Reported 95% accuracy is of the order obtainable from outcome-defining items (100% by a bare rule). We could not reconstruct the original predictor matrix, preprocessing, split or LSTM implementation, so we cannot establish that leakage caused the reported figure; the statement is that the figure is CONSISTENT with item-level circularity.", "n/a (inference)"),
 ("c) inference about published models", "Arab-region (Qasrawi et al.)", "Reported 83.3%/82.9% accuracies are of the same type of claim; item-level predictors were presumably included, but exact reconstruction was not possible.", "n/a (inference)"),
 ("c) inference about published models", "Pakistan (deep learning)", "On leakage-audited SDoH-only features an embedding MLP did not outperform RF (Phase 20); this does not test the original RNN-LSTM architecture on the original features.", "20_deep_learning_multiseed")]
T = pd.DataFrame(tiers, columns=["evidence_tier", "dataset", "statement", "source_phase"]); save_table(T, ENV, "S2_leakage_evidence_tiers.csv"); T

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_leakage_evidence_tiers.csv  (11 rows)


,evidence_tier,dataset,statement,source_phase
0,a) dataset fact,Pakistan,PHQ-9 item columns sum exactly to the supplied...,02_leakage_audit
1,a) dataset fact,Pakistan,The binary label equals (PHQ-9 score >= 10) in...,02_leakage_audit
2,a) dataset fact,Arab-region,PHQ-9 and GAD-7 items sum to the supplied scor...,02_leakage_audit
3,a) dataset fact,Arab-region,Rows with missing items were default-imputed t...,02_leakage_audit
4,b) reproduction (this study),Pakistan,Logistic regression on the 9 items -> label: 1...,02_leakage_audit
5,b) reproduction (this study),Arab-region,"Items -> label: 100% (depression), 99.7% (anxi...",02_leakage_audit
6,b) reproduction (this study),Arab-region,Missingness of the PHQ-9 block predictable fro...,02_leakage_audit / 05b
7,b) reproduction (this study),Pakistan,Validation-strategy contrast (ordinary vs grou...,17_validation_strategy_comparison
8,c) inference about published models,Pakistan (Zafar et al.),Reported 95% accuracy is of the order obtainab...,n/a (inference)
9,c) inference about published models,Arab-region (Qasrawi et al.),Reported 83.3%/82.9% accuracies are of the sam...,n/a (inference)


In [3]:
# Part B (optional): inspect the Pakistan oversampled file released with the original study.
# Column names below are copied verbatim from notebook 02 (raw file). If the oversampled file uses different names the cell prints its
# columns and stops WITHOUT running anything, so nothing can be silently mis-detected.
PHQ_COLS = ["Little interest or pleasure in doing things", "Feeling down, depressed, or hopeless",
            "Trouble falling or staying sleep or sleeping too much", "Feeling tired or having little energy",
            "Poor appetite or overeating", "Feeling badabout yourself that you are failure or have let yourself or your family down",
            "Trouble concentrating on things, such as reading the newspaper or watching television ",
            "Moving or speaking so slowly that other people could have Noticed. ",
            "Thoughts that you would be better off dead, or of hurting yourself"]
SCORE_COL, LABEL_COL = "Scalling", "Labelling"
fp = os.path.join(ENV["RAW"], ENV["CFG"]["raw_files"]["pakistan_oversampled"])
if not os.path.exists(fp):
    print("oversampled file not found:", fp)
else:
    from leakage_audit import deterministic_sum_check, naive_leaky_cv_accuracy
    osd = pd.read_csv(fp); print("shape:", osd.shape); print(list(osd.columns))
    missing = [c for c in PHQ_COLS + [SCORE_COL, LABEL_COL] if c not in osd.columns]
    if missing:
        print("Oversampled file does not contain these expected columns -> Part B skipped:", missing)
    else:
        yb = osd[LABEL_COL].astype(str).str.strip().str.lower().isin(["depressed", "1", "yes", "true"]).astype(int)
        mr, md_ = deterministic_sum_check(osd, PHQ_COLS, SCORE_COL)
        acc, sd = naive_leaky_cv_accuracy(osd[PHQ_COLS], yb, seed=SEED)
        thr_rule_acc = float((((osd[SCORE_COL] >= 10).astype(int)) == yb).mean())
        dup = int(osd.duplicated().sum())
        out = pd.DataFrame([dict(rows=len(osd), prevalence=yb.mean(), items_sum_match_rate=mr, mean_abs_diff=md_,
                                 bare_rule_score_ge_10_accuracy=thr_rule_acc, naive_items_cv_accuracy=acc, naive_cv_sd=sd,
                                 exact_duplicate_rows=dup, pct_exact_duplicates=100 * dup / len(osd))])
        save_table(out, ENV, "S2_pakistan_oversampled_inspection.csv"); print(out.T)

shape: (24660, 27)
['Age', 'Gestational Age', 'Number of sons ', 'Number of daughters', 'Total Number of Children', 'Gravida', 'Female Education', 'Husband Education', 'Working Status', 'Physical Health ', 'Previous Miscarriage', 'Current Appereance Acceptance', 'Family System', 'Male Gender Preference', 'Relationship with Mother in-law', 'Little interest or pleasure in doing things', 'Feeling down, depressed, or hopeless', 'Trouble falling or staying sleep or sleeping too much', 'Feeling tired or having little energy', 'Poor appetite or overeating', 'Feeling badabout yourself that you are failure or have let yourself or your family down', 'Trouble concentrating on things, such as reading the newspaper or watching television ', 'Moving or speaking so slowly that other people could have Noticed. ', 'Thoughts that you would be better off dead, or of hurting yourself', 'Scalling', 'Labelling', 'Sufficient Money for Basic Needs']
Saved: C:\maternal-mental-health-risk\reports/tables\revisio